# 第11章 代码教学：文本生成任务（机器翻译 + 问答）

> 目标：
> 1) 使用翻译模型 + 轻量文本生成模型完成翻译与问答
> 2) 对比解码策略（greedy / sampling）
> 3) 用最小脚本评估 BLEU / EM / F1


## 0. 环境准备

依赖：`transformers` `sacrebleu` `torch`。


In [1]:
# 可选：安装依赖
# !pip install -U transformers sacrebleu accelerate

import re
import numpy as np
import torch
import sacrebleu
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

np.random.seed(42)
torch.manual_seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device=', device)


C:\Users\250010108\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device= cpu


## 1. 加载翻译模型 + 轻量文本生成模型

翻译使用 `Helsinki-NLP/opus-mt-en-zh`，问答使用 `t5-small`。
如离线环境，请提前缓存或换成已存在模型。


In [2]:
MT_MODEL = 'Helsinki-NLP/opus-mt-en-zh'
QA_MODEL = 't5-small'

tok_mt = AutoTokenizer.from_pretrained(MT_MODEL)
model_mt = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL).to(device)
model_mt.eval()

tok_qa = AutoTokenizer.from_pretrained(QA_MODEL)
model_qa = AutoModelForSeq2SeqLM.from_pretrained(QA_MODEL).to(device)
model_qa.eval()

print('loaded mt', MT_MODEL)
print('loaded qa', QA_MODEL)


C:\Users\250010108\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\250010108\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-en-zh. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


C:\Users\250010108\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/258 [00:00<00:00, 21509.25it/s, Materializing param=final_logits_bias]

Loading weights:   0%|          | 1/258 [00:00<00:00, 3127.74it/s, Materializing param=final_logits_bias] 

...

Loading weights:  99%|█████████▉| 130/131 [00:00<00:00, 1650.40it/s, Materializing param=encoder.final_layer_norm.weight]

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 1654.74it/s, Materializing param=shared.weight]                  

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 1646.43it/s, Materializing param=shared.weight]

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 1638.00it/s, Materializing param=shared.weight]

loaded mt Helsinki-NLP/opus-mt-en-zh
loaded qa t5-small


## 2. 统一生成函数

通过 temperature / top_p 切换 greedy 与采样。


In [3]:
@torch.no_grad()
def gen(prompt, model, tok, max_new_tokens=96, temperature=0.0, top_p=1.0):
    inputs = tok(prompt, return_tensors='pt', truncation=True).to(device)
    do_sample = temperature > 0
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=max(temperature, 1e-5),
        top_p=top_p,
        num_beams=1 if do_sample else 4,
    )
    return tok.decode(outputs[0], skip_special_tokens=True).strip()

print(gen('translate English to Chinese: The model is robust.', model_mt, tok_mt))


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


英文译为中文:该模型很健全。


## 3. 任务 A：机器翻译（MT）


In [4]:
mt_data = [
    {"src":"Large language models are changing software development.","ref":"大型语言模型正在改变软件开发。"},
    {"src":"We use retrieval to reduce hallucination.","ref":"我们使用检索来减少幻觉。"},
    {"src":"The training loss decreases steadily.","ref":"训练损失稳定下降。"},
    {"src":"Quantization can reduce memory usage.","ref":"量化可以降低内存占用。"},
    {"src":"Agent systems need strong safety constraints.","ref":"智能体系统需要强安全约束。"},
]

def mt_prompt(src):
    return f"translate English to Chinese: {src}"

mt_greedy = [gen(mt_prompt(x['src']), model_mt, tok_mt, temperature=0.0) for x in mt_data]
mt_sample = [gen(mt_prompt(x['src']), model_mt, tok_mt, temperature=0.7, top_p=0.9) for x in mt_data]

for i, x in enumerate(mt_data):
    print(f"[{i}]", x['src'])
    print('ref=', x['ref'])
    print('greedy=', mt_greedy[i])
    print('sample=', mt_sample[i])
    print('-'*60)


[0] Large language models are changing software development.
ref= 大型语言模型正在改变软件开发。
greedy= 英文译为中文:大语言模式正在改变软件开发。
sample= 英文译为中文:大语言模式正在改变软件开发。
------------------------------------------------------------
[1] We use retrieval to reduce hallucination.
ref= 我们使用检索来减少幻觉。
greedy= 英文译为中文:我们使用检索来减少幻觉。
sample= 英文译为中文:我们使用检索减少幻觉。
------------------------------------------------------------
[2] The training loss decreases steadily.
ref= 训练损失稳定下降。
greedy= 英文译为中文:培训损失稳步下降。
sample= 英文译为中文:训练损失稳步下降。
------------------------------------------------------------
[3] Quantization can reduce memory usage.
ref= 量化可以降低内存占用。
greedy= 英文译为中文:量化可以减少记忆使用。
sample= 中文译为英文:量化可减少记忆使用。
------------------------------------------------------------
[4] Agent systems need strong safety constraints.
ref= 智能体系统需要强安全约束。
greedy= 英文译为中文:代理系统需要强大的安全限制。
sample= 英文译为中文:代理系统需要强大的安全限制
------------------------------------------------------------


## 4. BLEU 评估


In [5]:
refs = [x['ref'] for x in mt_data]
print('BLEU greedy=', sacrebleu.corpus_bleu(mt_greedy, [refs]).score)
print('BLEU sample=', sacrebleu.corpus_bleu(mt_sample, [refs]).score)


BLEU greedy= 0.0
BLEU sample= 0.0


## 5. 任务 B：问答（QA）


In [6]:
qa_data = [
    {"ctx":"LoRA keeps the base model frozen and trains low-rank adapters.","q":"What does LoRA train?","a":"low-rank adapters"},
    {"ctx":"RAG retrieves relevant passages before generation to improve factuality.","q":"Why is retrieval used in RAG?","a":"to improve factuality"},
    {"ctx":"Top-p sampling selects from the smallest token set whose cumulative probability exceeds p.","q":"How does top-p sampling work?","a":"cumulative probability exceeds p"},
    {"ctx":"KV cache stores previously computed key and value tensors for faster decoding.","q":"What is the role of KV cache?","a":"faster decoding"},
]

def qa_prompt(ctx, q):
    return (
        "Answer the question using the context. If missing, say 'unknown'.\n"
        f"context: {ctx}\nquestion: {q}\nanswer:"
    )

qa_pred = [gen(qa_prompt(x['ctx'], x['q']), model_qa, tok_qa, temperature=0.0) for x in qa_data]
for i, x in enumerate(qa_data):
    print(f"[{i}] Q=", x['q'])
    print('gold=', x['a'])
    print('pred=', qa_pred[i])
    print('-'*60)


[0] Q= What does LoRA train?
gold= low-rank adapters
pred= False
------------------------------------------------------------
[1] Q= Why is retrieval used in RAG?
gold= to improve factuality
pred= True
------------------------------------------------------------
[2] Q= How does top-p sampling work?
gold= cumulative probability exceeds p
pred= True
------------------------------------------------------------
[3] Q= What is the role of KV cache?
gold= faster decoding
pred= Answer the question using the context
------------------------------------------------------------


## 6. EM / Token-F1 评估


In [7]:
def norm(s):
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9一-龥\s]", " ", s)
    return re.sub(r"\s+", " ", s)

def em(p, g):
    return int(norm(p) == norm(g))

def f1(p, g):
    pt, gt = norm(p).split(), norm(g).split()
    if not pt and not gt: return 1.0
    if not pt or not gt: return 0.0
    bag = {}
    for t in pt: bag[t] = bag.get(t, 0) + 1
    inter = 0
    for t in gt:
        if bag.get(t, 0) > 0:
            inter += 1; bag[t] -= 1
    if inter == 0: return 0.0
    pre = inter / len(pt)
    rec = inter / len(gt)
    return 2 * pre * rec / (pre + rec)

ems = [em(p, x['a']) for p, x in zip(qa_pred, qa_data)]
f1s = [f1(p, x['a']) for p, x in zip(qa_pred, qa_data)]
print('EM=', 100*np.mean(ems))
print('F1=', 100*np.mean(f1s))


EM= 0.0
F1= 0.0


## 7. 解码策略对比（同一 QA 样本）


In [8]:
x = qa_data[0]
p = qa_prompt(x['ctx'], x['q'])
for cfg in [(0.0,1.0),(0.4,0.9),(0.8,0.95)]:
    print(cfg, '->', gen(p, model_mt, tok_mt, temperature=cfg[0], top_p=cfg[1]))


(0.0, 1.0) -> 使用上下文回答问题。 如果丢失了, 请说“ 未知 ” 。 上下文 : LoRA 使基础模型冻结并训练低级适应器。 问题 : LoRA 火车是做什么的? 回答 :


(0.4, 0.9) -> 回答使用上下文的问题。 如果丢失, 则请使用“ 未知” 。 上下文 : LoRA 使基础模型冻结并训练低级别适应者。 问题 : LoRA 列车是做什么的? 回答 :


(0.8, 0.95) -> 使用上下文回答问题。 缺少的话, 请填写 : LoRA 控制着基础模型并训练低级别适应器。 问题: LoRA 将培训什么? 答案:


## 8. 统一任务路由


In [9]:
def run_task(task_type, **kw):
    if task_type == 'translation':
        return gen(mt_prompt(kw['source']), model_mt, tok_mt, temperature=kw.get('temperature',0.0))
    if task_type == 'qa':
        return gen(qa_prompt(kw['context'], kw['question']), model_qa, tok_qa, temperature=kw.get('temperature',0.0))
    raise ValueError(task_type)

print(run_task('translation', source='RAG combines retrieval and generation.'))
print(run_task('qa', context='MoE activates a subset of experts.', question='How does MoE save computation?'))


英文译为中文:RAG结合检索和生成。
True


## 9. 练习
1) 增加摘要任务并复用 gen()。
2) 引入术语表约束，比较 BLEU 与一致性。
3) 加入检索上下文，比较 QA 的 EM/F1。
